In [ ]:
!pip install torch transformers datasets pennylane pennylane-lightning matplotlib accelerate tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 34.5 MB/s eta 0:00:00


# DeepSeek

In [ ]:
import sys
import tempfile

# Fix for __main__.__file__ issue in Transformers (needed in notebooks)
dummy_file = tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False)
dummy_file.write("# Dummy module file for transformers")
dummy_file.close()
sys.modules['__main__'].__file__ = dummy_file.name

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, PreTrainedModel, PretrainedConfig,
    GenerationMixin
)
from datasets import load_dataset
from torch.utils.data import DataLoader
import time
import re
from typing import Optional, Tuple

# ============================================================================
# 1. Custom Configuration
# ============================================================================
class SuperpositionConfig(PretrainedConfig):
    model_type = "superposition"
    def __init__(
        self,
        base_model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        K=4,
        num_superposition_layers=-1,   # -1 means all layers
        hidden_size=2048,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.base_model_name = base_model_name
        self.K = K
        self.num_superposition_layers = num_superposition_layers
        self.hidden_size = hidden_size

# ============================================================================
# 2. Entanglement Module (Multi-head attention over hypotheses)
# ============================================================================
class HypothesisEntanglement(nn.Module):
    def __init__(self, hidden_dim, K, num_heads=8, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.K = K
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        assert self.head_dim * num_heads == hidden_dim, "hidden_dim must be divisible by num_heads"

        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5

    def forward(self, h):
        """
        h: [B, T, K, D]  (hypotheses dimension)
        returns: [B, T, K, D] after attention over hypotheses
        """
        B, T, K, D = h.shape
        h_flat = h.view(B*T, K, D)

        q = self.q_proj(h_flat)
        k = self.k_proj(h_flat)
        v = self.v_proj(h_flat)

        q = q.view(B*T, K, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B*T, K, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B*T, K, self.num_heads, self.head_dim).transpose(1, 2)

        attn_weights = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.dropout(attn_weights)

        attn_output = torch.matmul(attn_weights, v)
        attn_output = attn_output.transpose(1, 2).contiguous().view(B*T, K, D)
        attn_output = self.out_proj(attn_output)

        out = h_flat + attn_output
        return out.view(B, T, K, D)

# ============================================================================
# 3. Interference Module (Gating to amplify/suppress hypotheses)
# ============================================================================
class HypothesisInterference(nn.Module):
    def __init__(self, hidden_dim, K):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Linear(hidden_dim // 4, 1)
        )

    def forward(self, h):
        scores = self.gate(h)          # [B,T,K,1]
        gate = torch.sigmoid(scores)
        return h * gate

# ============================================================================
# 4. Collapse Module (Learned measurement)
# ============================================================================
class HypothesisCollapse(nn.Module):
    def __init__(self, hidden_dim, K):
        super().__init__()
        self.measure = nn.Linear(hidden_dim, 1)

    def forward(self, h):
        weights = torch.softmax(self.measure(h), dim=2)   # [B,T,K,1]
        collapsed = torch.sum(h * weights, dim=2)         # [B,T,D]
        return collapsed, weights.squeeze(-1)

# ============================================================================
# 5. Layer Wrapper (Adds superposition to a single transformer layer)
# ============================================================================
class SuperpositionLayerWrapper(nn.Module):
    def __init__(self, original_layer, hidden_dim, K, num_heads=8, dropout=0.1):
        super().__init__()
        self.original_layer = original_layer
        self.entanglement = HypothesisEntanglement(hidden_dim, K, num_heads, dropout)
        self.interference = HypothesisInterference(hidden_dim, K)
        self.collapse = HypothesisCollapse(hidden_dim, K)
        self.K = K
        self.hidden_dim = hidden_dim

    def forward(self, hidden_states, attention_mask=None, position_ids=None, past_key_value=None, output_attentions=False, use_cache=False, **kwargs):
        # Forward through original layer
        if past_key_value is not None:
            layer_output = self.original_layer(
                hidden_states,
                attention_mask=attention_mask,
                position_ids=position_ids,
                past_key_value=past_key_value,
                output_attentions=output_attentions,
                use_cache=use_cache,
                **kwargs
            )
        else:
            layer_output = self.original_layer(
                hidden_states,
                attention_mask=attention_mask,
                position_ids=position_ids,
                output_attentions=output_attentions,
                use_cache=use_cache,
                **kwargs
            )

        # Extract hidden states and other outputs
        if isinstance(layer_output, tuple):
            h = layer_output[0]
            other_outputs = layer_output[1:]
        else:
            h = layer_output
            other_outputs = ()

        B, T, D = h.shape

        # Expand to K hypotheses
        h_expanded = h.unsqueeze(2).expand(-1, -1, self.K, -1)

        # Entanglement (attention over hypotheses)
        h_entangled = self.entanglement(h_expanded)

        # Interference (gating)
        h_interfered = self.interference(h_entangled)

        # Collapse back to single hypothesis
        h_collapsed, _ = self.collapse(h_interfered)

        # Residual connection (original output + collapsed superposition)
        h_out = h + h_collapsed

        return (h_out,) + other_outputs

# ============================================================================
# 6. Full Model with Superposition
# ============================================================================
class SuperpositionModel(PreTrainedModel, GenerationMixin):
    config_class = SuperpositionConfig
    _no_split_modules = ["SuperpositionLayerWrapper"]

    def __init__(self, config, base_model=None):
        super().__init__(config)
        self.config = config
        if base_model is None:
            self.base = AutoModelForCausalLM.from_pretrained(
                config.base_model_name,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto"
            )
        else:
            self.base = base_model

        self.base_config = self.base.config
        self.hidden_size = self.base_config.hidden_size

        # Freeze base model parameters (only superposition modules are trained)
        for param in self.base.parameters():
            param.requires_grad = False

        # Determine which layers to wrap
        num_layers = len(self.base.model.layers)
        if config.num_superposition_layers == -1:
            layers_to_wrap = list(range(num_layers))
        else:
            layers_to_wrap = list(range(num_layers))[-config.num_superposition_layers:]

        # Replace selected layers with wrapped versions
        for idx in layers_to_wrap:
            self.base.model.layers[idx] = SuperpositionLayerWrapper(
                self.base.model.layers[idx],
                self.hidden_size,
                config.K,
                num_heads=8,
                dropout=0.1
            )

        self.lm_head = self.base.lm_head
        self.generation_config = self.base.generation_config
        if self.generation_config.pad_token_id is None:
            self.generation_config.pad_token_id = self.base.config.pad_token_id or self.base.config.eos_token_id

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        position_ids=None,
        past_key_values=None,
        inputs_embeds=None,
        labels=None,
        output_hidden_states=None,
        return_dict=None,
        **kwargs
    ):
        outputs = self.base.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
            **kwargs
        )

        hidden_states = outputs.last_hidden_state if hasattr(outputs, "last_hidden_state") else outputs[0]
        logits = self.lm_head(hidden_states)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        if not return_dict:
            output = (logits,) + outputs[1:]
            return ((loss,) + output) if loss is not None else output

        return {
            "loss": loss,
            "logits": logits,
            "past_key_values": outputs.past_key_values,
            "hidden_states": outputs.hidden_states,
            "attentions": outputs.attentions,
        }

# ============================================================================
# 7. Data Preparation (Full GSM8K)
# ============================================================================
def prepare_dataset(tokenizer, max_length=512):
    dataset = load_dataset("gsm8k", "main")
    train_dataset = dataset["train"]
    test_dataset = dataset["test"]

    def preprocess(example):
        text = f"Question: {example['question']}\nAnswer: {example['answer']}"
        enc = tokenizer(text, truncation=True, padding="max_length", max_length=max_length)
        enc["labels"] = enc["input_ids"].copy()
        return enc

    train_dataset = train_dataset.map(preprocess, remove_columns=train_dataset.column_names)
    test_dataset = test_dataset.map(preprocess, remove_columns=test_dataset.column_names)

    train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    return train_dataset, test_dataset

# ============================================================================
# 8. Training Loop
# ============================================================================
def train(model, train_loader, optimizer, num_epochs=3, device="cuda"):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs["loss"]
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# ============================================================================
# 9. Evaluation (Extract numeric answer and compare)
# ============================================================================
def extract_answer(text):
    match = re.search(r"####\s*([\d.,]+)", text)
    if match:
        return match.group(1).replace(",", "")
    numbers = re.findall(r"\b\d+\b", text)
    return numbers[-1] if numbers else None

def evaluate(model, test_dataset, tokenizer, device="cuda", max_new_tokens=128):
    model.eval()
    correct = 0
    total_time = 0
    for example in test_dataset:
        prompt = f"Question: {example['question']}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        true_answer = extract_answer(example["answer"])

        torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        torch.cuda.synchronize()
        elapsed = time.time() - start
        total_time += elapsed

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred_answer = extract_answer(generated_text)

        if pred_answer == true_answer:
            correct += 1

    accuracy = correct / len(test_dataset)
    avg_time = total_time / len(test_dataset)
    return accuracy, avg_time

# ============================================================================
# 10. Main Execution
# ============================================================================
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
    tokenizer.pad_token = tokenizer.eos_token

    # Configuration
    config = SuperpositionConfig(
        base_model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        K=4,
        num_superposition_layers=-1,   # wrap all layers
        hidden_size=2048
    )

    # Load base model (shared between baseline and superposition model)
    base_model = AutoModelForCausalLM.from_pretrained(
        config.base_model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto"
    )
    base_model.eval()

    # Create superposition model (sharing the same base)
    model = SuperpositionModel(config, base_model=base_model).to(device)

    # Prepare dataset (use full training set for better results)
    print("Loading and tokenizing dataset...")
    train_dataset, test_dataset = prepare_dataset(tokenizer, max_length=512)
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

    # Optimizer: only train superposition modules (base is frozen)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=1e-4)

    # Training
    print("\nTraining superposition model...")
    train(model, train_loader, optimizer, num_epochs=3, device=device)

    # Evaluation (on a subset of test set for speed; change to len(test_dataset) for full)
    test_subset = test_dataset.select(range(50))
    print("\nEvaluating superposition model...")
    sup_acc, sup_time = evaluate(model, test_subset, tokenizer, device=device)
    print(f"Superposition Model - Accuracy: {sup_acc:.4f}, Avg Time: {sup_time:.4f}s")

    # Baseline evaluation (load fresh base model)
    base_model_eval = AutoModelForCausalLM.from_pretrained(
        config.base_model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto"
    )
    base_model_eval.eval()
    print("\nEvaluating base model...")
    base_acc, base_time = evaluate(base_model_eval, test_subset, tokenizer, device=device)
    print(f"Base Model - Accuracy: {base_acc:.4f}, Avg Time: {base_time:.4f}s")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]